In [14]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import joblib
from pathlib import Path

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions, RunningMode

In [15]:
# Paths
NEW_VIDEO_PATH = Path("../data/raw/test_video_data/new_video2.mp4")
MODEL_PATH    = Path("../data/models/pose_landmarker_lite.task")
OUTPUT_CSV    = Path("../data/raw/test_tabular_data/test_angles_dataset2.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete.")
print(f"   Model        : {MODEL_PATH.resolve()}")
print(f"   Video source : {NEW_VIDEO_PATH.resolve()}")
print(f"   Output CSV   : {OUTPUT_CSV.resolve()}")

✅ Setup complete.
   Model        : C:\Users\USER\Desktop\AI_PROJECT_2\data\models\pose_landmarker_lite.task
   Video source : C:\Users\USER\Desktop\AI_PROJECT_2\data\raw\test_video_data\new_video2.mp4
   Output CSV   : C:\Users\USER\Desktop\AI_PROJECT_2\data\raw\test_tabular_data\test_angles_dataset2.csv


In [16]:
import sys
sys.path.insert(0, "..")
from src.shared_utils import calculate_angle, LANDMARKS, get_coords, get_z, extract_angles

# Sanity check
test_angle = calculate_angle((0, 0), (1, 0), (1, 1))
print(f"Shared utilities loaded. Sanity check (expect 90): {test_angle}°")


✅ Angle function works. Test angle (expected 90°): 90.0°


In [17]:
# LANDMARKS, get_coords, get_z, extract_angles are imported from src/shared_utils.py above.
print("   Features per frame: 10 joint angles + 3 torso rotation = 13 total.")


✅ Landmark helpers and extract_angles() ready.
   Features per frame: 10 joint angles + 3 torso rotation = 13 total.


In [18]:
def process_video(video_path, options):
    """
    Process a single video file and return a list of rows,
    where each row represents one frame's angles.

    Args:
        video_path          : Path to the video file.
        options             : PoseLandmarkerOptions (initialized outside)

    Returns:
        List of dicts, one per valid frame.
    """
    rows = []
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"   ⚠️  Could not open: {video_path.name}")
        return rows

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30  # fallback

    prev_angles = None
    frame_number = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Convert frame to MediaPipe Image
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            # Timestamp in milliseconds (required for VIDEO mode)
            timestamp_ms = int((frame_number / fps) * 1000)

            # Run pose detection
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]  # first (and usually only) person
                angles    = extract_angles(landmarks)

                # --- Angular velocities (change in angle per second) ---
                if prev_angles is not None:
                    angular_velocities = {
                        f"{key}_velocity": round(abs(angles[key] - prev_angles[key]) * fps, 2)
                        for key in angles
                    }
                else:
                    # First valid frame: velocity is 0
                    angular_velocities = {f"{key}_velocity": 0.0 for key in angles}

                row = {
                    "frame_number"        : frame_number,
                    **angles,
                    **angular_velocities,
                }
                rows.append(row)
                prev_angles = angles

            frame_number += 1

    cap.release()
    return rows


print("✅ Video processing function ready.")
print("   Outputs per frame: frame_number + 10 angles + 10 angular velocities")

✅ Video processing function ready.
   Outputs per frame: frame_number + 10 angles + 10 angular velocities


In [19]:
def build_dataset(video_path):
    """
    Process one new unseen video and return an unlabeled DataFrame.
    """
    all_rows = []

    # Initialize PoseLandmarker options once
    base_options = mp_python.BaseOptions(model_asset_path=str(MODEL_PATH))
    options = PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )

    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    print(f"\n🎞️ Processing: {video_path.name} ...", end=" ")
    rows = process_video(video_path, options)
    all_rows.extend(rows)
    print(f"{len(rows)} frames extracted.")

    df = pd.DataFrame(all_rows)
    return df


# --- Run it ---
print("🚀 Starting dataset creation...\n")
df_raw = build_dataset(NEW_VIDEO_PATH)

print(f"\n✅ Done! Total frames collected: {len(df_raw)}")
print(f"   Shape : {df_raw.shape}")
print(f"   Columns: {list(df_raw.columns)}")

🚀 Starting dataset creation...


🎞️ Processing: new_video2.mp4 ... 145 frames extracted.

✅ Done! Total frames collected: 145
   Shape : (145, 27)
   Columns: ['frame_number', 'left_elbow_angle', 'right_elbow_angle', 'left_shoulder_angle', 'right_shoulder_angle', 'left_hip_angle', 'right_hip_angle', 'left_knee_angle', 'right_knee_angle', 'left_ankle_angle', 'right_ankle_angle', 'shoulder_z_diff', 'hip_z_diff', 'torso_rotation', 'left_elbow_angle_velocity', 'right_elbow_angle_velocity', 'left_shoulder_angle_velocity', 'right_shoulder_angle_velocity', 'left_hip_angle_velocity', 'right_hip_angle_velocity', 'left_knee_angle_velocity', 'right_knee_angle_velocity', 'left_ankle_angle_velocity', 'right_ankle_angle_velocity', 'shoulder_z_diff_velocity', 'hip_z_diff_velocity', 'torso_rotation_velocity']


In [20]:
# --- Save raw dataset ---
df_raw.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Raw dataset saved to: {OUTPUT_CSV}")
print(f"   Shape: {df_raw.shape}\n")

# --- Sanity Checks ---
angle_cols    = [col for col in df_raw.columns if "angle" in col and "velocity" not in col]
velocity_cols = [col for col in df_raw.columns if "velocity" in col]

# 1. Preview
print("=" * 60)
print("📋 First 5 rows:")
display(df_raw.head())

# 2. Null check
print("=" * 60)
null_counts = df_raw.isnull().sum()
if null_counts.sum() == 0:
    print("✅ No null values found.")
else:
    print("⚠️  Null values detected:")
    display(null_counts[null_counts > 0])

# 3. Angle range check → DROP out-of-range rows (mirrors training cleaning)
print("=" * 60)
before = len(df_raw)
valid_mask = df_raw[angle_cols].apply(lambda col: (col >= 0) & (col <= 180)).all(axis=1)
df_raw = df_raw[valid_mask].reset_index(drop=True)
dropped = before - len(df_raw)

if dropped == 0:
    print("✅ All angles within [0°, 180°]. No rows dropped.")
else:
    print(f"⚠️  Dropped {dropped} frames with out-of-range angles. Remaining: {len(df_raw)}")

# 4. Basic statistics
print("=" * 60)
print("📈 Angle columns statistics:")
display(df_raw[angle_cols].describe())

✅ Raw dataset saved to: ..\data\raw\test_tabular_data\test_angles_dataset2.csv
   Shape: (145, 27)

📋 First 5 rows:


,frame_number,left_elbow_angle,right_elbow_angle,left_shoulder_angle,right_shoulder_angle,left_hip_angle,right_hip_angle,left_knee_angle,right_knee_angle,left_ankle_angle,...,right_shoulder_angle_velocity,left_hip_angle_velocity,right_hip_angle_velocity,left_knee_angle_velocity,right_knee_angle_velocity,left_ankle_angle_velocity,right_ankle_angle_velocity,shoulder_z_diff_velocity,hip_z_diff_velocity,torso_rotation_velocity
0,0,14.52,51.16,124.98,118.32,92.84,94.47,137.36,134.51,138.01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
1,1,22.28,93.26,128.24,108.65,85.29,86.17,129.75,132.90,140.94,...,290.1,226.5,249.0,228.3,48.3,87.9,485.1,1.27,0.01,1.28
2,2,33.09,86.43,127.03,112.21,81.43,82.48,126.42,130.11,139.95,...,106.8,115.8,110.7,99.9,83.7,29.7,16.5,0.12,0.00,0.12
3,3,31.56,75.04,129.64,115.78,72.84,76.01,120.92,125.53,136.38,...,107.1,257.7,194.1,165.0,137.4,107.1,54.9,0.57,0.09,0.48
4,4,35.17,97.13,129.39,114.78,66.78,70.10,115.39,115.14,137.18,...,30.0,181.8,177.3,165.9,311.7,24.0,247.8,0.21,0.13,0.34


✅ No null values found.
✅ All angles within [0°, 180°]. No rows dropped.
📈 Angle columns statistics:


,left_elbow_angle,right_elbow_angle,left_shoulder_angle,right_shoulder_angle,left_hip_angle,right_hip_angle,left_knee_angle,right_knee_angle,left_ankle_angle,right_ankle_angle
count,145.000000,145.00000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000
mean,71.666345,94.41069,123.379379,106.897724,75.837310,78.002759,115.293448,112.445793,139.496759,140.588069
std,52.168526,53.06694,17.922807,32.072493,50.913863,50.908576,30.865892,30.648815,5.410410,9.489943
min,3.240000,1.84000,81.490000,8.210000,20.460000,24.350000,78.120000,75.750000,124.330000,112.270000
25%,22.070000,38.76000,114.480000,94.400000,36.300000,35.510000,91.840000,86.380000,137.330000,135.390000
50%,58.830000,106.19000,129.870000,115.140000,54.360000,55.230000,101.840000,97.150000,140.250000,140.390000
75%,116.780000,138.96000,136.910000,132.060000,112.840000,114.700000,139.740000,134.940000,142.500000,144.750000
max,171.430000,179.34000,147.150000,141.960000,179.900000,179.990000,175.430000,173.170000,151.060000,169.330000


In [21]:
# Load the scaler used during training
SCALER_PATH = Path("../data/models/scaler.pkl")
scaler = joblib.load(SCALER_PATH)

print("✅ Scaler loaded successfully.")
print(f"   Scaler path: {SCALER_PATH.resolve()}")

✅ Scaler loaded successfully.
   Scaler path: C:\Users\USER\Desktop\AI_PROJECT_2\data\models\scaler.pkl


In [22]:
metadata_cols = ["frame_number"]
feature_cols  = [col for col in df_raw.columns if col not in metadata_cols]

# --- Velocity noise cap (mirrors training preprocessing) ---
velocity_cols = [col for col in feature_cols if "velocity" in col]
df_raw[velocity_cols] = df_raw[velocity_cols].clip(upper=3000)
print(f"✅ Velocity values capped at 3000 deg/s ({len(velocity_cols)} columns).")

# --- Feature count guard ---
assert len(feature_cols) == scaler.n_features_in_, (
    f"Feature mismatch! Inference has {len(feature_cols)} features, "
    f"but scaler expects {scaler.n_features_in_}."
)

# --- Apply scaler ---
X_scaled = scaler.transform(df_raw[feature_cols])

df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
df_scaled.insert(0, "frame_number", df_raw["frame_number"].values)

print("✅ Scaling complete.")
print(f"   Shape: {df_scaled.shape}")

✅ Velocity values capped at 3000 deg/s (13 columns).
✅ Scaling complete.
   Shape: (145, 27)


In [23]:
# Save scaled dataset for inference
OUTPUT_SCALED_PATH = Path("../data/processed/test_tabular_data/new_video_features_scaled2.csv")
OUTPUT_SCALED_PATH.parent.mkdir(parents=True, exist_ok=True)

df_scaled.to_csv(OUTPUT_SCALED_PATH, index=False)

print("✅ Scaled dataset saved.")
print(f"   Path: {OUTPUT_SCALED_PATH.resolve()}")

✅ Scaled dataset saved.
   Path: C:\Users\USER\Desktop\AI_PROJECT_2\data\processed\test_tabular_data\new_video_features_scaled2.csv


In [24]:
import joblib

# Load trained models
EXERCISE_SVM_MODEL_PATH    = Path("../data/models/svm_exercise_tuned.pkl")
CORRECTNESS_SVM_MODEL_PATH = Path("../data/models/svm_correctness_tuned.pkl")
EXERCISE_RF_MODEL_PATH        = Path("../data/models/rf_exercise_tuned.pkl")
CORRECTNESS_RF_MODEL_PATH     = Path("../data/models/rf_correctness_tuned.pkl")

exercise_model = joblib.load(EXERCISE_SVM_MODEL_PATH)
correctness_model = joblib.load(CORRECTNESS_SVM_MODEL_PATH)
exercise_rf_model = joblib.load(EXERCISE_RF_MODEL_PATH)
correctness_rf_model = joblib.load(CORRECTNESS_RF_MODEL_PATH)

print("✅ Models loaded successfully.")
print(f"   Exercise model   : {EXERCISE_SVM_MODEL_PATH.resolve()}")
print(f"   Correctness model: {CORRECTNESS_SVM_MODEL_PATH.resolve()}")
print(f"   Exercise RF model: {EXERCISE_RF_MODEL_PATH.resolve()}")
print(f"   Correctness RF model: {CORRECTNESS_RF_MODEL_PATH.resolve()}")

✅ Models loaded successfully.
   Exercise model   : C:\Users\USER\Desktop\AI_PROJECT_2\data\models\svm_exercise_tuned.pkl
   Correctness model: C:\Users\USER\Desktop\AI_PROJECT_2\data\models\svm_correctness_tuned.pkl
   Exercise RF model: C:\Users\USER\Desktop\AI_PROJECT_2\data\models\rf_exercise_tuned.pkl
   Correctness RF model: C:\Users\USER\Desktop\AI_PROJECT_2\data\models\rf_correctness_tuned.pkl


In [25]:
metadata_cols = ["frame_number"]
feature_cols  = [col for col in df_scaled.columns if col not in metadata_cols]

X = df_scaled[feature_cols]

# --- SVM Predictions ---
df_scaled["pred_exercise_svm"]    = exercise_model.predict(X)
df_scaled["pred_correctness_svm"] = correctness_model.predict(X)

# --- Random Forest Predictions ---
df_scaled["pred_exercise_rf"]    = exercise_rf_model.predict(X)
df_scaled["pred_correctness_rf"] = correctness_rf_model.predict(X)

print("✅ Predictions completed.")
display(df_scaled[[
    "frame_number",
    "pred_exercise_svm", "pred_correctness_svm",
    "pred_exercise_rf",  "pred_correctness_rf"
]].head(10))

✅ Predictions completed.


,frame_number,pred_exercise_svm,pred_correctness_svm,pred_exercise_rf,pred_correctness_rf
0,0,0,1,1,1
1,1,3,1,0,1
2,2,1,1,1,1
3,3,3,1,1,1
4,4,3,1,3,1
5,5,0,1,3,1
6,6,3,1,3,1
7,7,3,1,3,1
8,8,1,1,3,1
9,9,0,1,3,0


In [26]:
# --- Aggregate predictions for single video (majority vote) ---

def majority_vote(series):
    return series.value_counts().idxmax()

overall_predictions = {
    "exercise_svm": majority_vote(df_scaled["pred_exercise_svm"]),
    "correctness_svm": majority_vote(df_scaled["pred_correctness_svm"]),
    "exercise_rf": majority_vote(df_scaled["pred_exercise_rf"]),
    "correctness_rf": majority_vote(df_scaled["pred_correctness_rf"]),
}

print("✅ Overall prediction (entire video):")
for k, v in overall_predictions.items():
    print(f"   {k}: {v}")

✅ Overall prediction (entire video):
   exercise_svm: 3
   correctness_svm: 1
   exercise_rf: 3
   correctness_rf: 1
